# Notebook 17 — Higher-Order Transition Memory + Shuffle Baseline

**prime-numbers-lab**

This notebook continues the Notebook 16 transition-operator path and tests whether
prime-gap residual structure contains **higher-order transition memory** beyond a
first-order Markov transition operator.

Notebook 17 keeps the locked repo template:

```text
17_higher_order_transition_memory_shuffle_baseline/
  figures/
  data/
  docs/
  tex/
17_higher_order_transition_memory_shuffle_baseline_export.zip
```

Core question:

> Does a second-order transition model retain measurable structure after comparing
> real prime-gap residual transitions against a shuffle baseline?

Outputs include transition matrices, lift maps, mutual-information diagnostics,
shuffle comparisons, CSV summaries, Markdown notes, and a compact LaTeX section.

In [ ]:
# ============================================================
# Notebook 17 locked-template setup
# ============================================================

import os
import math
import json
import zipfile
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "17_higher_order_transition_memory_shuffle_baseline"
OUTDIR = Path(NOTEBOOK_ID)
FIGDIR = OUTDIR / "figures"
DATADIR = OUTDIR / "data"
DOCDIR = OUTDIR / "docs"
TEXDIR = OUTDIR / "tex"

for d in [OUTDIR, FIGDIR, DATADIR, DOCDIR, TEXDIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 9423
rng = np.random.default_rng(SEED)

X_MAX = 2_000_000
WINDOWS = 13
Z_BINS = np.linspace(0.0, 6.0, 61)
Z_CENTERS = 0.5 * (Z_BINS[:-1] + Z_BINS[1:])
N_STATES = 8

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True,
    "font.size": 12,
})

print("Notebook:", NOTEBOOK_ID)
print("Output directory:", OUTDIR.resolve())

## 1. Prime data and normalized gap residuals

We compute primes up to `X_MAX`, prime gaps `g_n = p_{n+1} - p_n`, and normalized
gaps

\[
z_n = \frac{g_n}{\log(p_n)}.
\]

Notebook 17 discretizes these normalized gaps into quantile states, then compares:

- first-order transition operator: `P(s_{n+1} | s_n)`
- empirical second-order transition operator: `P(s_{n+2} | s_n, s_{n+1})`
- Markov-predicted second-order operator from first-order transitions
- shuffle baseline that preserves one-point distribution but destroys ordering

In [ ]:
# ============================================================
# Prime generator and gap features
# ============================================================

def sieve_primes(n: int) -> np.ndarray:
    """Return all primes <= n using a bytearray sieve."""
    if n < 2:
        return np.array([], dtype=np.int64)
    sieve = bytearray(b"\x01") * (n + 1)
    sieve[0:2] = b"\x00\x00"
    limit = int(n**0.5)
    for p in range(2, limit + 1):
        if sieve[p]:
            start = p * p
            step = p
            sieve[start:n+1:step] = b"\x00" * (((n - start) // step) + 1)
    return np.fromiter((i for i, v in enumerate(sieve) if v), dtype=np.int64)

primes = sieve_primes(X_MAX)
p = primes[:-1]
gaps = np.diff(primes).astype(float)
logp = np.log(p)
z = gaps / logp

# Remove tiny first-prime edge and keep finite values.
mask = np.isfinite(z) & (p >= 101)
p = p[mask]
gaps = gaps[mask]
logp = logp[mask]
z = z[mask]

dataset_summary = {
    "notebook_id": NOTEBOOK_ID,
    "x_max": X_MAX,
    "prime_count": int(len(primes)),
    "gap_count_used": int(len(z)),
    "p_min": int(p.min()),
    "p_max": int(p.max()),
    "mean_gap": float(np.mean(gaps)),
    "mean_logp": float(np.mean(logp)),
    "mean_normalized_gap_z": float(np.mean(z)),
    "std_normalized_gap_z": float(np.std(z)),
    "seed": SEED,
}

pd.DataFrame([dataset_summary]).to_csv(DATADIR / "17_dataset_summary.csv", index=False)
dataset_summary

## 2. Windowing and state discretization

To test memory at multiple scales, we create logarithmic windows and assign each
normalized gap to a quantile state. Quantile states avoid a trivial sparsity
problem: every state has support before transition operators are estimated.

In [ ]:
# ============================================================
# Scale windows and quantile states
# ============================================================

edges = np.geomspace(max(101, int(p.min())), int(p.max()), WINDOWS + 1).astype(int)
window_rows = []
window_index = np.full(len(p), -1, dtype=int)

for i in range(WINDOWS):
    lo, hi = edges[i], edges[i + 1]
    if i == WINDOWS - 1:
        m = (p >= lo) & (p <= hi)
    else:
        m = (p >= lo) & (p < hi)
    window_index[m] = i
    if m.sum() > 0:
        window_rows.append({
            "window_index": i,
            "x_min": int(lo),
            "x_max": int(hi),
            "window_midpoint": float(np.sqrt(lo * hi)),
            "count": int(m.sum()),
            "mean_z": float(np.mean(z[m])),
            "std_z": float(np.std(z[m])),
            "q50_z": float(np.quantile(z[m], 0.50)),
            "q90_z": float(np.quantile(z[m], 0.90)),
            "q99_z": float(np.quantile(z[m], 0.99)),
        })

window_summary = pd.DataFrame(window_rows)
window_summary.to_csv(DATADIR / "17_window_summary.csv", index=False)

# Global quantile states.
quantile_edges = np.quantile(z, np.linspace(0, 1, N_STATES + 1))
quantile_edges[0] = -np.inf
quantile_edges[-1] = np.inf
states = np.digitize(z, quantile_edges[1:-1], right=False).astype(int)

state_summary = pd.DataFrame({
    "state": np.arange(N_STATES),
    "q_edge_low": quantile_edges[:-1],
    "q_edge_high": quantile_edges[1:],
    "count": np.bincount(states, minlength=N_STATES),
})
state_summary.to_csv(DATADIR / "17_state_summary.csv", index=False)

window_summary.head(), state_summary

## 3. Transition operators

We estimate:

\[
P_{ij}=\Pr(s_{n+1}=j\mid s_n=i),
\]

and a two-step empirical operator

\[
P^{(2)}_{ik}=\Pr(s_{n+2}=k\mid s_n=i).
\]

The Markov baseline is

\[
\widehat P^{(2)} = P^2.
\]

The difference `empirical_two_step - Markov_two_step` is a direct test for
higher-order transition memory.

In [ ]:
# ============================================================
# Transition operators
# ============================================================

def row_normalize(M, eps=1e-12):
    M = np.asarray(M, dtype=float)
    row_sum = M.sum(axis=1, keepdims=True)
    return np.divide(M, row_sum + eps, out=np.zeros_like(M), where=row_sum > 0)

def transition_operator(s, n_states=N_STATES):
    C = np.zeros((n_states, n_states), dtype=float)
    for a, b in zip(s[:-1], s[1:]):
        C[a, b] += 1
    return row_normalize(C), C

def two_step_operator(s, n_states=N_STATES):
    C = np.zeros((n_states, n_states), dtype=float)
    for a, c in zip(s[:-2], s[2:]):
        C[a, c] += 1
    return row_normalize(C), C

def second_order_tensor(s, n_states=N_STATES):
    C = np.zeros((n_states, n_states, n_states), dtype=float)
    for a, b, c in zip(s[:-2], s[1:-1], s[2:]):
        C[a, b, c] += 1
    T = np.zeros_like(C)
    denom = C.sum(axis=2, keepdims=True)
    np.divide(C, denom, out=T, where=denom > 0)
    return T, C

P1, C1 = transition_operator(states)
P2_emp, C2_emp = two_step_operator(states)
P2_markov = P1 @ P1
T2, Cabc = second_order_tensor(states)

delta_P2 = P2_emp - P2_markov
lift = np.divide(P2_emp, P2_markov + 1e-12)

pd.DataFrame(P1).to_csv(DATADIR / "17_transition_operator_P.csv", index=False)
pd.DataFrame(P2_emp).to_csv(DATADIR / "17_empirical_two_step_operator_P2.csv", index=False)
pd.DataFrame(P2_markov).to_csv(DATADIR / "17_markov_predicted_two_step_operator_P2.csv", index=False)
pd.DataFrame(delta_P2).to_csv(DATADIR / "17_two_step_delta_operator.csv", index=False)
pd.DataFrame(lift).to_csv(DATADIR / "17_two_step_lift_operator.csv", index=False)

operator_metrics = {
    "first_order_entropy": float(-np.sum(P1[P1 > 0] * np.log(P1[P1 > 0])) / N_STATES),
    "two_step_l1_delta": float(np.mean(np.abs(delta_P2))),
    "two_step_l2_delta": float(np.sqrt(np.mean(delta_P2**2))),
    "max_abs_two_step_delta": float(np.max(np.abs(delta_P2))),
    "max_lift": float(np.max(lift[np.isfinite(lift)])),
    "min_lift": float(np.min(lift[np.isfinite(lift)])),
}
pd.DataFrame([operator_metrics]).to_csv(DATADIR / "17_operator_metrics.csv", index=False)
operator_metrics

In [ ]:
# ============================================================
# Figure helpers
# ============================================================

figure_paths = []

def savefig(name: str):
    path = FIGDIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    figure_paths.append(path)
    plt.show()
    return path

def plot_heatmap(M, title, xlabel, ylabel, colorbar_label, filename, vmin=None, vmax=None):
    plt.figure(figsize=(9, 7))
    im = plt.imshow(M, aspect="auto", origin="lower", vmin=vmin, vmax=vmax)
    plt.colorbar(im, label=colorbar_label)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    savefig(filename)

In [ ]:
# ============================================================
# Figures: first-order and two-step transition structure
# ============================================================

plot_heatmap(
    P1,
    "First-order transition operator",
    "next state",
    "current state",
    "P(s_next | s)",
    "17_transition_operator_P_heatmap.png",
)

plot_heatmap(
    P2_emp,
    "Empirical two-step operator",
    "state after two steps",
    "current state",
    "P(s_{n+2} | s_n)",
    "17_empirical_two_step_operator_heatmap.png",
)

plot_heatmap(
    delta_P2,
    "Two-step delta: empirical minus Markov baseline",
    "state after two steps",
    "current state",
    "delta probability",
    "17_two_step_operator_delta_heatmap.png",
)

plot_heatmap(
    np.clip(lift, 0, 3),
    "Two-step lift: empirical / Markov baseline",
    "state after two steps",
    "current state",
    "lift, clipped at 3",
    "17_two_step_transition_lift_heatmap.png",
)

## 4. Shuffle baseline

We compare real ordering against shuffled state sequences. The shuffle preserves
state counts but removes sequential structure. If two-step memory is real, the
real-vs-shuffle lift and delta should remain nonzero.

In [ ]:
# ============================================================
# Shuffle baseline
# ============================================================

N_SHUFFLES = 64

shuffle_rows = []
shuffle_P2 = []
shuffle_delta_norms = []

for k in range(N_SHUFFLES):
    s_shuf = states.copy()
    rng.shuffle(s_shuf)
    P1_s, _ = transition_operator(s_shuf)
    P2_s, _ = two_step_operator(s_shuf)
    delta_s = P2_s - (P1_s @ P1_s)
    shuffle_P2.append(P2_s)
    shuffle_delta_norms.append(np.sqrt(np.mean(delta_s**2)))
    shuffle_rows.append({
        "shuffle_id": k,
        "two_step_l1_delta": float(np.mean(np.abs(delta_s))),
        "two_step_l2_delta": float(np.sqrt(np.mean(delta_s**2))),
        "max_abs_two_step_delta": float(np.max(np.abs(delta_s))),
    })

shuffle_P2_mean = np.mean(shuffle_P2, axis=0)
real_vs_shuffle = P2_emp - shuffle_P2_mean
real_vs_shuffle_lift = np.divide(P2_emp, shuffle_P2_mean + 1e-12)

shuffle_summary = pd.DataFrame(shuffle_rows)
shuffle_summary.to_csv(DATADIR / "17_shuffle_residual_summary.csv", index=False)
pd.DataFrame(shuffle_P2_mean).to_csv(DATADIR / "17_shuffle_two_step_baseline.csv", index=False)
pd.DataFrame(real_vs_shuffle).to_csv(DATADIR / "17_real_minus_shuffle_two_step_operator.csv", index=False)

real_shuffle_metrics = {
    "real_two_step_l2_delta": float(operator_metrics["two_step_l2_delta"]),
    "shuffle_mean_two_step_l2_delta": float(shuffle_summary["two_step_l2_delta"].mean()),
    "shuffle_std_two_step_l2_delta": float(shuffle_summary["two_step_l2_delta"].std()),
    "real_minus_shuffle_mean_abs": float(np.mean(np.abs(real_vs_shuffle))),
    "real_minus_shuffle_l2": float(np.sqrt(np.mean(real_vs_shuffle**2))),
    "real_over_shuffle_max_lift": float(np.max(real_vs_shuffle_lift)),
}
pd.DataFrame([real_shuffle_metrics]).to_csv(DATADIR / "17_lift_metrics.csv", index=False)
real_shuffle_metrics

In [ ]:
# ============================================================
# Figures: real-vs-shuffle two-step structure
# ============================================================

plot_heatmap(
    real_vs_shuffle,
    "Real minus shuffle two-step operator",
    "state after two steps",
    "current state",
    "delta probability",
    "17_real_vs_shuffle_two_step_delta_heatmap.png",
)

plot_heatmap(
    np.clip(real_vs_shuffle_lift, 0, 3),
    "Real / shuffle two-step lift",
    "state after two steps",
    "current state",
    "lift, clipped at 3",
    "17_real_vs_shuffle_two_step_lift_heatmap.png",
)

plt.figure()
plt.hist(shuffle_summary["two_step_l2_delta"], bins=18, alpha=0.75, label="shuffle baseline")
plt.axvline(operator_metrics["two_step_l2_delta"], linestyle="--", linewidth=2, label="real sequence")
plt.title("Real two-step memory versus shuffle baseline")
plt.xlabel("two-step L2 delta")
plt.ylabel("frequency")
plt.legend()
savefig("17_real_vs_shuffle_two_step_l2.png")

## 5. Mutual information and lag structure

We measure empirical mutual information across lags:

\[
I(s_n; s_{n+\ell})
=\sum_{ij} p_{ij}\log\frac{p_{ij}}{p_i p_j}.
\]

A faster shuffle decay or lower shuffle baseline supports ordered transition memory.

In [ ]:
# ============================================================
# Mutual information by lag
# ============================================================

def mutual_information_discrete(a, b, n_states=N_STATES, eps=1e-12):
    C = np.zeros((n_states, n_states), dtype=float)
    for x, y in zip(a, b):
        C[x, y] += 1
    Pxy = C / max(C.sum(), 1.0)
    Px = Pxy.sum(axis=1, keepdims=True)
    Py = Pxy.sum(axis=0, keepdims=True)
    denom = Px @ Py
    mask = Pxy > 0
    return float(np.sum(Pxy[mask] * np.log((Pxy[mask] + eps) / (denom[mask] + eps))))

lags = np.arange(1, 21)
mi_rows = []

# One shuffled baseline sequence for lag profile plus average over multiple shuffles.
shuffle_mi_all = []
for k in range(N_SHUFFLES):
    s_shuf = states.copy()
    rng.shuffle(s_shuf)
    shuffle_mi_all.append([mutual_information_discrete(s_shuf[:-lag], s_shuf[lag:]) for lag in lags])
shuffle_mi_all = np.asarray(shuffle_mi_all)

for idx, lag in enumerate(lags):
    real_mi = mutual_information_discrete(states[:-lag], states[lag:])
    mi_rows.append({
        "lag": int(lag),
        "real_mi": real_mi,
        "shuffle_mean_mi": float(shuffle_mi_all[:, idx].mean()),
        "shuffle_std_mi": float(shuffle_mi_all[:, idx].std()),
        "excess_mi": float(real_mi - shuffle_mi_all[:, idx].mean()),
    })

mi_df = pd.DataFrame(mi_rows)
mi_df.to_csv(DATADIR / "17_mutual_information_summary.csv", index=False)

plt.figure()
plt.plot(mi_df["lag"], mi_df["real_mi"], marker="o", label="real")
plt.plot(mi_df["lag"], mi_df["shuffle_mean_mi"], marker="o", label="shuffle mean")
plt.fill_between(
    mi_df["lag"],
    mi_df["shuffle_mean_mi"] - mi_df["shuffle_std_mi"],
    mi_df["shuffle_mean_mi"] + mi_df["shuffle_std_mi"],
    alpha=0.2,
    label="shuffle ±1 std",
)
plt.title("Mutual information versus lag")
plt.xlabel("lag")
plt.ylabel("mutual information")
plt.legend()
savefig("17_mutual_information_vs_lag.png")

plt.figure()
plt.plot(mi_df["lag"], mi_df["excess_mi"], marker="o")
plt.axhline(0, linestyle="--")
plt.title("Excess mutual information over shuffle baseline")
plt.xlabel("lag")
plt.ylabel("real MI - shuffle mean MI")
savefig("17_excess_mutual_information_vs_lag.png")

mi_df.head()

## 6. Windowed higher-order memory

The global operator can hide scale variation, so we recompute transition-memory
metrics inside logarithmic windows. Each window estimates a first-order and
two-step operator, then measures two-step lift away from a Markov baseline.

In [ ]:
# ============================================================
# Windowed higher-order transition metrics
# ============================================================

window_metric_rows = []
window_delta_maps = []
window_lift_maps = []

for _, row in window_summary.iterrows():
    wi = int(row["window_index"])
    m = window_index == wi
    s_w = states[m]
    if len(s_w) < 20:
        continue

    P1_w, _ = transition_operator(s_w)
    P2_w, _ = two_step_operator(s_w)
    D_w = P2_w - (P1_w @ P1_w)
    L_w = np.divide(P2_w, (P1_w @ P1_w) + 1e-12)

    window_delta_maps.append(D_w)
    window_lift_maps.append(np.clip(L_w, 0, 3))

    window_metric_rows.append({
        "window_index": wi,
        "window_midpoint": float(row["window_midpoint"]),
        "count": int(row["count"]),
        "two_step_l1": float(np.mean(np.abs(D_w))),
        "two_step_l2": float(np.sqrt(np.mean(D_w**2))),
        "max_abs_delta": float(np.max(np.abs(D_w))),
        "mean_lift": float(np.mean(L_w[np.isfinite(L_w)])),
        "max_lift": float(np.max(L_w[np.isfinite(L_w)])),
    })

window_metrics = pd.DataFrame(window_metric_rows)
window_metrics.to_csv(DATADIR / "17_windowed_higher_order_residual_metrics.csv", index=False)

plt.figure()
plt.plot(window_metrics["window_midpoint"], window_metrics["two_step_l1"], marker="o", label="L1 two-step residual")
plt.plot(window_metrics["window_midpoint"], window_metrics["two_step_l2"], marker="o", label="L2 two-step residual")
plt.xscale("log")
plt.title("Windowed two-step residual metrics")
plt.xlabel("window midpoint x")
plt.ylabel("residual norm")
plt.legend()
savefig("17_windowed_two_step_residual_metrics.png")

plt.figure()
plt.plot(window_metrics["window_midpoint"], window_metrics["mean_lift"], marker="o", label="mean lift")
plt.plot(window_metrics["window_midpoint"], window_metrics["max_lift"], marker="o", label="max lift")
plt.xscale("log")
plt.title("Windowed higher-order lift metrics")
plt.xlabel("window midpoint x")
plt.ylabel("lift")
plt.legend()
savefig("17_windowed_information_structure.png")

window_metrics

In [ ]:
# ============================================================
# Windowed heatmap summaries
# ============================================================

if window_delta_maps:
    # Convert each window map to mean absolute delta by transition-pair state.
    delta_stack = np.stack([np.abs(M) for M in window_delta_maps], axis=0)
    lift_stack = np.stack(window_lift_maps, axis=0)

    # Flatten state-pair columns.
    pair_labels = [f"{i}->{j}" for i in range(N_STATES) for j in range(N_STATES)]
    delta_flat = delta_stack.reshape(delta_stack.shape[0], -1)
    lift_flat = lift_stack.reshape(lift_stack.shape[0], -1)

    pd.DataFrame(delta_flat, columns=pair_labels).to_csv(DATADIR / "17_windowed_transition_delta_flat.csv", index=False)
    pd.DataFrame(lift_flat, columns=pair_labels).to_csv(DATADIR / "17_windowed_transition_lift_flat.csv", index=False)

    plt.figure(figsize=(12, 6))
    im = plt.imshow(delta_flat, aspect="auto", origin="lower")
    plt.colorbar(im, label="abs two-step delta")
    plt.title("Windowed transition delta heatmap")
    plt.xlabel("state-pair transition")
    plt.ylabel("window index")
    plt.xticks(np.arange(0, len(pair_labels), 8), pair_labels[::8], rotation=45, ha="right")
    savefig("17_windowed_transition_delta_heatmap.png")

    plt.figure(figsize=(12, 6))
    im = plt.imshow(lift_flat, aspect="auto", origin="lower", vmin=0, vmax=3)
    plt.colorbar(im, label="two-step lift, clipped at 3")
    plt.title("Windowed transition lift heatmap")
    plt.xlabel("state-pair transition")
    plt.ylabel("window index")
    plt.xticks(np.arange(0, len(pair_labels), 8), pair_labels[::8], rotation=45, ha="right")
    savefig("17_windowed_transition_lift_heatmap.png")

## 7. Top conditional-chain residuals

The strongest residual chains identify which two-step paths differ most from the
first-order Markov prediction.

In [ ]:
# ============================================================
# Top residual transition chains
# ============================================================

top_rows = []
for i in range(N_STATES):
    for k in range(N_STATES):
        top_rows.append({
            "from_state": i,
            "to_state_after_two_steps": k,
            "empirical_P2": float(P2_emp[i, k]),
            "markov_P2": float(P2_markov[i, k]),
            "delta": float(delta_P2[i, k]),
            "abs_delta": float(abs(delta_P2[i, k])),
            "lift": float(lift[i, k]),
        })

top_df = pd.DataFrame(top_rows).sort_values("abs_delta", ascending=False)
top_df.to_csv(DATADIR / "17_top_two_step_residual_transitions.csv", index=False)

plt.figure(figsize=(11, 6))
plot_df = top_df.head(20).copy()
labels = [f"{int(r.from_state)}→{int(r.to_state_after_two_steps)}" for _, r in plot_df.iterrows()]
plt.bar(labels, plot_df["abs_delta"])
plt.xticks(rotation=45, ha="right")
plt.title("Top two-step residual transition masses")
plt.xlabel("state n -> state n+2")
plt.ylabel("|empirical - Markov|")
savefig("17_top_two_step_residual_transitions.png")

# Conditional-chain tensor residual: P(c | a,b) minus P(c | b)
chain_rows = []
for a in range(N_STATES):
    for b in range(N_STATES):
        for c in range(N_STATES):
            chain_rows.append({
                "state_n": a,
                "state_n_plus_1": b,
                "state_n_plus_2": c,
                "empirical_P_c_given_ab": float(T2[a, b, c]),
                "first_order_P_c_given_b": float(P1[b, c]),
                "delta": float(T2[a, b, c] - P1[b, c]),
                "abs_delta": float(abs(T2[a, b, c] - P1[b, c])),
                "support_count_ab": int(Cabc[a, b, :].sum()),
            })

chain_df = pd.DataFrame(chain_rows)
chain_df = chain_df[chain_df["support_count_ab"] >= 10].sort_values("abs_delta", ascending=False)
chain_df.to_csv(DATADIR / "17_conditional_two_step_chain_residuals.csv", index=False)

plt.figure(figsize=(12, 6))
plot_chain = chain_df.head(20)
labels = [f"{int(r.state_n)}→{int(r.state_n_plus_1)}→{int(r.state_n_plus_2)}" for _, r in plot_chain.iterrows()]
plt.bar(labels, plot_chain["abs_delta"])
plt.xticks(rotation=45, ha="right")
plt.title("Top conditional-chain residuals")
plt.xlabel("state chain")
plt.ylabel("|P(c|a,b)-P(c|b)|")
savefig("17_top_conditional_chain_residuals.png")

top_df.head(10), chain_df.head(10)

## 8. Interpretation numbers

This cell prints compact, paper-friendly numbers for this notebook.

In [ ]:
# ============================================================
# Interpretation summary
# ============================================================

mi1 = float(mi_df.loc[mi_df["lag"] == 1, "real_mi"].iloc[0])
mi1_shuffle = float(mi_df.loc[mi_df["lag"] == 1, "shuffle_mean_mi"].iloc[0])
mi2 = float(mi_df.loc[mi_df["lag"] == 2, "real_mi"].iloc[0])
mi2_shuffle = float(mi_df.loc[mi_df["lag"] == 2, "shuffle_mean_mi"].iloc[0])

interpretation = {
    "two_step_l2_delta": operator_metrics["two_step_l2_delta"],
    "shuffle_mean_two_step_l2_delta": real_shuffle_metrics["shuffle_mean_two_step_l2_delta"],
    "real_to_shuffle_l2_ratio": float(operator_metrics["two_step_l2_delta"] / (real_shuffle_metrics["shuffle_mean_two_step_l2_delta"] + 1e-12)),
    "max_abs_two_step_delta": operator_metrics["max_abs_two_step_delta"],
    "mean_abs_real_minus_shuffle": real_shuffle_metrics["real_minus_shuffle_mean_abs"],
    "lag1_real_mi": mi1,
    "lag1_shuffle_mi": mi1_shuffle,
    "lag1_excess_mi": mi1 - mi1_shuffle,
    "lag2_real_mi": mi2,
    "lag2_shuffle_mi": mi2_shuffle,
    "lag2_excess_mi": mi2 - mi2_shuffle,
    "top_two_step_residual": float(top_df["abs_delta"].iloc[0]),
    "top_conditional_chain_residual": float(chain_df["abs_delta"].iloc[0]) if len(chain_df) else np.nan,
}

interpretation_df = pd.DataFrame([interpretation])
interpretation_df.to_csv(DATADIR / "17_interpretation_summary.csv", index=False)

for k, v in interpretation.items():
    print(f"{k}: {v:.6g}" if isinstance(v, float) else f"{k}: {v}")

interpretation

## 9. Markdown + LaTeX exports

This locked-template cell writes a short interpretation document and a LaTeX section.

In [ ]:
# ============================================================
# Documentation and LaTeX exports
# ============================================================

figures_md = "\n\n## Figures\n\n"
for idx, fig in enumerate(sorted(FIGDIR.glob("*.png")), start=1):
    title = fig.stem.replace("_", " ").title()
    figures_md += f"### Figure {idx} — {title}\n\n"
    figures_md += f"![Figure {idx}](../figures/{fig.name})\n\n"

summary_lines = [
    f"# {NOTEBOOK_ID}: higher-order transition memory + shuffle baseline",
    "",
    "## Purpose",
    "",
    "Notebook 17 tests whether normalized prime-gap residual states carry higher-order transition memory.",
    "It compares empirical two-step transitions against a first-order Markov prediction and a shuffle baseline.",
    "",
    "## Key numbers",
    "",
]
for k, v in interpretation.items():
    if isinstance(v, float):
        summary_lines.append(f"- `{k}`: `{v:.6g}`")
    else:
        summary_lines.append(f"- `{k}`: `{v}`")

summary_lines += [
    "",
    "## Interpretation",
    "",
    "A nonzero two-step residual means the empirical transition sequence is not completely described by the first-order operator.",
    "The shuffle baseline preserves the one-point state distribution while destroying ordering, so real-vs-shuffle excess indicates ordered memory rather than marginal frequency alone.",
    "Windowed diagnostics localize where higher-order residual structure concentrates across scale.",
    "",
    figures_md,
]

(DOCDIR / "17_interpretation_summary.md").write_text("\n".join(summary_lines), encoding="utf-8")

tex = rf"""
\section{{Notebook 17: Higher-Order Transition Memory and Shuffle Baseline}}

Notebook 17 tests whether normalized prime-gap residual states contain higher-order
transition memory beyond a first-order Markov operator.  Let $s_n$ denote a
quantile-discretized state of the normalized gap
\[
z_n = \frac{{p_{{n+1}}-p_n}}{{\log p_n}}.
\]
The first-order operator is
\[
P_{{ij}}=\Pr(s_{{n+1}}=j\mid s_n=i),
\]
while the empirical two-step operator is
\[
P^{{(2)}}_{{ik}}=\Pr(s_{{n+2}}=k\mid s_n=i).
\]
The Markov prediction is $\widehat{{P}}^{{(2)}}=P^2$, giving the residual
\[
\Delta^{{(2)}} = P^{{(2)}}-\widehat{{P}}^{{(2)}}.
\]

For this run, the mean two-step $L_2$ residual is
\[
\|\Delta^{{(2)}}\|_2 = {interpretation['two_step_l2_delta']:.6g},
\]
compared with a shuffle-baseline mean of
\[
{interpretation['shuffle_mean_two_step_l2_delta']:.6g}.
\]
The lag-one excess mutual information is
\[
I_1^{{\rm real}}-I_1^{{\rm shuffle}} =
{interpretation['lag1_excess_mi']:.6g},
\]
and the lag-two excess mutual information is
\[
I_2^{{\rm real}}-I_2^{{\rm shuffle}} =
{interpretation['lag2_excess_mi']:.6g}.
\]
These diagnostics quantify ordered transition memory after controlling for the
one-point state distribution.
"""
(TEXDIR / "17_higher_order_transition_memory_shuffle_baseline.tex").write_text(tex, encoding="utf-8")

print("Wrote docs and TeX:")
print(DOCDIR / "17_interpretation_summary.md")
print(TEXDIR / "17_higher_order_transition_memory_shuffle_baseline.tex")

## 10. Manifest and optional Colab download

The final cell writes a manifest and creates the standard root-level export zip.

Locked-template standard:

```python
# from google.colab import files
# files.download("17_higher_order_transition_memory_shuffle_baseline_export.zip")
```

In [ ]:
# ============================================================
# Manifest + locked-template export zip
# ============================================================

manifest_rows = []
for subdir in [FIGDIR, DATADIR, DOCDIR, TEXDIR]:
    for path in sorted(subdir.glob("*")):
        if path.is_file():
            manifest_rows.append({
                "notebook_id": NOTEBOOK_ID,
                "relative_path": str(path),
                "folder": path.parent.name,
                "filename": path.name,
                "size_bytes": path.stat().st_size,
            })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(DATADIR / "17_outputs_manifest.csv", index=False)

EXPORT_ZIP = Path(f"{NOTEBOOK_ID}_export.zip")
with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTDIR.rglob("*")):
        if path.is_file():
            zf.write(path, arcname=str(path))

print("Export zip created:", EXPORT_ZIP)
print("Files in manifest:", len(manifest))
print("Zip size bytes:", EXPORT_ZIP.stat().st_size)

# Optional: download outputs bundle (template standard)
# from google.colab import files
# files.download("17_higher_order_transition_memory_shuffle_baseline_export.zip")